##  TechMind — Exploración y Preparación del Dataset **Coursera 2021**

#### Equipo tejONEs

#### 01_exploracion_dataset_coursera.ipynb

💡**Dataset**: [Kaggle - Coursera Courses dataset 2021](https://www.kaggle.com/datasets/khusheekapoor/coursera-courses-dataset-2021)


- El proceso general que siguen este pipeline es:
    - [x]  Carga y normalización del dataset
    - [x]  Limpieza de texto (HTML, URLs, duplicados, filtro de mínimo 100 palabras)
    - [x]  Mapeo de categorías por palabras clave (Backend, Frontend, Data Science, DevOps, Bases de Datos, Mobile, Cloud)
    - [x]  Balanceo de datos (máximo 50 registros por categoría)
    - [x]  Traducción a español (título, texto) con manejo de errores visible
    - [x]  Exportación del dataset final, sin columnas duplicadas, en carpeta `procesados/`

## Importaciones

In [1]:
import os
import re
from pathlib import Path

import nltk
import pandas as pd
from deep_translator import GoogleTranslator
from nltk.corpus import stopwords
from tqdm.notebook import tqdm

# Descargar recursos de NLP
nltk.download('stopwords', quiet=True)
spanish_stopwords = set(stopwords.words('spanish'))

## Configuración

In [2]:
def find_project_root(start: Path) -> Path:
    """Encuentra la raíz del proyecto desde el directorio actual o sus padres."""
    for candidate in [start, *start.parents]:
        if (candidate / 'data_science').exists() and (candidate / 'README.md').exists():
            return candidate
    return start


base_dir = Path.cwd().resolve()
project_root = find_project_root(base_dir)

CARPETA_DATA = str((project_root / 'data_science' / 'data').resolve())
CARPETA_CRUDOS = str((project_root / 'data_science' / 'data' / 'crudos').resolve())
CARPETA_PROCESADOS = str((project_root / 'data_science' / 'data' / 'procesados').resolve())


print(f'✅ Ruta de procesados existe: {Path(CARPETA_PROCESADOS).exists()}')
print(f'✅ Ruta de crudos existe: {Path(CARPETA_CRUDOS).exists()}')
print(f'✅ Ruta de datos existe: {Path(CARPETA_DATA).exists()}')
print(f'📂 Datos procesados: {CARPETA_PROCESADOS}')
print(f'📂 Datos crudos: {CARPETA_CRUDOS}')
print(f'📁 Proyecto local: {CARPETA_DATA}')

✅ Ruta de procesados existe: True
✅ Ruta de crudos existe: True
✅ Ruta de datos existe: True
📂 Datos procesados: C:\Users\neo_p\Documents\G9-LATAM-Team-25\data_science\data\procesados
📂 Datos crudos: C:\Users\neo_p\Documents\G9-LATAM-Team-25\data_science\data\crudos
📁 Proyecto local: C:\Users\neo_p\Documents\G9-LATAM-Team-25\data_science\data


# 1.Coursera Courses dataset 2021

##  1.1 Carga de datos y normalización

In [3]:
file_path = f'{CARPETA_CRUDOS}/dataset_coursera.csv'
# Carga segura con alternativa de codificación (encoding fallback) en caso de error
try:
    df = pd.read_csv(file_path, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(file_path, encoding='latin1')

# Renombrar columnas para cumplir estrictamente con el esquema de la base de datos
df = df.rename(columns={
    'Course Name': 'titulo',
    'Course Description': 'texto',
    'Skills': 'categoria_original',
    'University': 'autor',
})[['titulo', 'texto', 'categoria_original', 'autor']].copy()

# 'tipo' respeta el esquema del equipo: "texto" (tecleado a mano, vía POST /contenido)

# o "articulo" (documento con fuente, vía carga de archivo). Este dataset son

# descripciones de cursos - documentos, no texto tecleado a mano - así que correspondedisplay(df.sample(3))

# 'articulo'.print("\n--- Muestra aleatoria de los datos cargados ---")

df['tipo'] = 'articulo'

print(f"✅ Datos cargados correctamente. Filas totales: {len(df)}")

✅ Datos cargados correctamente. Filas totales: 3522


## 1.2 Limpieza inicial del texto

In [4]:
def clean_html_urls(text):
    text = re.sub(r'<[^>]+>', ' ', str(text))
    return re.sub(r'http\S+', '', text)

df['texto'] = df['texto'].apply(clean_html_urls)
df['titulo'] = df['titulo'].apply(clean_html_urls)

# Eliminar duplicados basados en la descripción del curso
initial_count = len(df)
df = df.drop_duplicates(subset='texto').reset_index(drop=True)
print(f"Duplicados eliminados: {initial_count - len(df)} (Filas restantes: {len(df)})")

# Filtro de calidad: mínimo 100 palabras en la columna 'texto'. Un umbral de caracteres
# es demasiado permisivo — 20 caracteres son apenas 3-4 palabras, casi no filtra nada.
initial_count = len(df)
df = df[df['texto'].str.split().str.len() >= 100].reset_index(drop=True)
print(f"Textos con menos de 100 palabras eliminados: {initial_count - len(df)} (Filas restantes: {len(df)})")

print("✅ Limpieza de HTML, URLs, duplicados y textos cortos completada.")

Duplicados eliminados: 125 (Filas restantes: 3397)
Textos con menos de 100 palabras eliminados: 698 (Filas restantes: 2699)
✅ Limpieza de HTML, URLs, duplicados y textos cortos completada.


## 1.3. Mapeo de categorías (Regex por palabras clave)


In [5]:
CATEGORY_KEYWORDS = {
    'Backend': ['backend','java', 'spring', 'c#', 'php', 'node.js', 'django', 'perl', 'ruby', 'scala', 'clojure', 'rust', 'haskell', 'elixir', 'earlang', 'flask', 'typescript', 'node', 'nodejs', 'laravel', 'api', 'rest', 'backend', 'csharp', 'dotnet', 'blockchain', 'cplusplus', 'graphql', 'kafka', 'solr', 'rabbitMQ', 'nginx', 'openresty', 'nestjs', 'firebase', '.net', 'rails'],
    'Bases de Datos': ['data base','data bases','bd','sql', 'mysql', 'mongodb', 'postgresql', 'redis', 'oracledb', 'cassandra', 'couchdb', 'hive', 'realm', 'mariadb', 'cockroachdb', 'elasticsearch', 'sqlite', 'mssql', 'sql server', 'sqlserver', 'cosmos db', 'database', 'bases de datos', 'nosql'],
    'Cloud': ['cloud','aws', 'azure', 'oraclecloud', 'oci', 'googlecloud', 'gcp', 'nube', 'cloud', 'virtual machine'],
    'Data Science': ['data science','python', 'pandas', 'machine learning', 'excel','bigquery', 'llm', 'r', 'ia', 'deep learning', 'tensorflow', 'pytorch', 'numpy', 'seaborn', 'matplotlib', 'opencv', 'scikitlearn', 'scikit learn','scikit-learn', 'd3js', 'chartjs', 'canvasjs', 'kibana', 'grafana', 'artificial intelligence', 'data mining','data analytics', 'data analysis','data modeling','powerbi','tableau'],
    'DevOps': ['devops','docker', 'kubernetes', 'ci/cd', 'devops', 'git', 'github', 'jenkins', 'bash', 'travisci', 'circleci', 'containers'],
    'Frontend': ['frontend','javascript', 'html', 'html5', 'css', 'css3', 'react', 'angular', 'angularjs', 'vue', 'vuejs', 'scratch', 'frontend', 'svelte', 'backbonejs', 'bootstrap', 'vuetify', 'pug', 'gulp', 'sass', 'redux', 'webpack', 'babel', 'tailwind', 'materialize', 'bulma', 'gtk', 'qt', 'quasar', 'wxwidgets', 'wx widgets', 'ember', 'blazor wasm','blazor webassembly'],
    'Mobile': ['objectivec', 'objective-c', 'android', 'ios', 'flutter', 'kotlin', 'swift', 'dart', 'nativescript', 'xamarin', 'reactnative', 'react native', 'ionic', 'apachecordova', 'mobile', 'multiplatform']
}

# Aplanamos el diccionario para facilitar la búsqueda
category_mapping = {}
for category, keywords in CATEGORY_KEYWORDS.items():
    for kw in keywords:
        category_mapping[kw.lower()] = category

def _find_category_in_text(text):
    text = str(text).lower()
    # Ordenar de mayor a menor longitud para evitar que claves cortas coincidan dentro de largas
    sorted_keys = sorted(category_mapping.items(), key=lambda x: -len(x[0]))
    for key, category in sorted_keys:
        # \b asegura la coincidencia exacta de la palabra
        pattern = r'\b' + re.escape(key) + r'\b'
        if re.search(pattern, text):
            return category, key
    return None, None

def assign_category_from_multiple_cols(row):
    # 1. Intentar mapear desde 'titulo' y 'texto' combinados
    combined_text = (str(row['titulo']) + ' ' + str(row['texto'])).lower()
    category, keyword = _find_category_in_text(combined_text)
    if category is not None:
        return category, keyword

    # 2. Si no se encuentra, intentar mapear desde 'categoria_original'
    category_orig = str(row['categoria_original']).lower()
    category, keyword = _find_category_in_text(category_orig)
    if category is not None:
        return category, keyword

    return None, None

# Agregamos una columna 'palabra_clave' para saber la palabra que se encontro para categorizar
results = df.apply(assign_category_from_multiple_cols, axis=1)
df[['categoria', 'palabra_clave']] = pd.DataFrame(results.tolist(), index=df.index)

# Diagnóstico: tasa de coincidencia
total_rows = len(df)
matched_rows = df['categoria'].notna().sum()
print(f" Categorías mapeadas con éxito: {matched_rows} de {total_rows} ({matched_rows/total_rows*100:.1f}%)" if total_rows > 0 else "No hay datos para procesar.")

if total_rows > 0 and (matched_rows / total_rows) < 0.3:
    print("⚠️ ALERTA: La tasa de coincidencia es menor al 30%. Revisa el diccionario de mapeo.")

df = df.dropna(subset=['categoria']).reset_index(drop=True)

print("\n✅ Categorías asignadas.")
print("--- Distribución resultante tras el mapeo ---")
print(df['categoria'].value_counts())

print("\n--- Muestra de validación (Keyword vs Categoría) ---")
display(df[['titulo','texto', 'palabra_clave', 'categoria','categoria_original']].sample(min(10, len(df))))

 Categorías mapeadas con éxito: 1136 de 2699 (42.1%)

✅ Categorías asignadas.
--- Distribución resultante tras el mapeo ---
categoria
Data Science      652
Backend           132
Mobile             90
Frontend           79
Cloud              74
Bases de Datos     66
DevOps             43
Name: count, dtype: int64

--- Muestra de validación (Keyword vs Categoría) ---


,titulo,texto,palabra_clave,categoria,categoria_original
102,Serve Scikit-Learn Models for Deployment with ...,This is a hands-on project on serving your sci...,machine learning,Data Science,project mine Logistic Regression modeling s...
328,Information Visualization: Advanced Techniques,This course aims to introduce learners to Adva...,kibana,Data Science,Information Visualization stack (abstract dat...
287,Building Smart Business Assistants with IBM Wa...,This is two-hour project-based course teaches ...,cloud,Cloud,interactivity natural language preview Acco...
762,Bitcoin and Cryptocurrency Technologies,To really understand what is special about Bit...,blockchain,Backend,consensus decision-making Java Programming B...
85,Autodesk Certified Professional: Inventor for ...,Prove to potential employers that you�re up to...,vue,Frontend,autodesk inventor Java Programming Mechanica...
797,Big Data Applications: Machine Learning at Scale,Machine learning is transforming the world aro...,machine learning,Data Science,Ensemble Learning Gradient Boosting Algorith...
894,Draw and Style Custom Letters with Inkscape,"By the end of this project, you will be able t...",node,Backend,image quality json path (variable) vector g...
769,Android Graphics with OpenGL ES,This course will cover the fundamentals of Ope...,android,Mobile,shading rendering (computer graphics) opengl...
235,Build a Firebase Android Application,This 1.5 hours class is the Android counterpar...,firebase,Backend,NoSQL Data Structures Android Development M...
115,A Complete Reinforcement Learning System (Caps...,"In this final course, you will put together yo...",machine learning,Data Science,Q-Learning Human Learning function approxima...


##  1.4. Balanceo de datos (Máximo 50 registros por categoría)

In [6]:
df = pd.concat([
    group.sample(min(len(group), 50), random_state=42)
    for _, group in df.groupby('categoria')
]).reset_index(drop=True)

print("✅ Datos balanceados (Máximo 50 registros por categoría).")

✅ Datos balanceados (Máximo 50 registros por categoría).


##  1.5. Traducción y procesamiento NLP

In [7]:
tqdm.pandas()

translation_errors = []

def translate_to_spanish(text):
    try:
        return GoogleTranslator(source='en', target='es').translate(text[:1500])
    except Exception as e:
        translation_errors.append(type(e).__name__)
        return ""

# Traduce título y texto principal al español
df['titulo_es'] = df['titulo'].progress_apply(translate_to_spanish)
df['texto_es'] = df['texto'].progress_apply(translate_to_spanish)

if translation_errors:
    from collections import Counter
    print(f"⚠️ {len(translation_errors)} traducciones fallaron. Tipos de error: {Counter(translation_errors)}")

def clean_nlp(text):
    # Quita signos de puntuación y pasa a minúsculas
    text = re.sub(r'[^\w\sáéíóúñ]', ' ', text.lower())
    # Elimina stopwords en español y palabras muy cortas
    return ' '.join([word for word in text.split() if word not in spanish_stopwords and len(word) > 2])

df['texto_limpio'] = df['texto_es'].apply(clean_nlp)

# Elimina filas donde la traducción del texto falló
initial_count = len(df)
df = df[df['texto_es'].str.strip() != ''].reset_index(drop=True)
print(f"\nTraducciones fallidas eliminadas: {initial_count - len(df)} (Filas restantes: {len(df)})")

# Guarda un respaldo
df.to_csv(f'{CARPETA_PROCESADOS}/translation_backup.csv', index=False)
print("✅ Traducción completada y respaldo guardado.")

# Reemplaza las columnas originales y descarta las auxiliares
df['titulo'] = df['titulo_es']
df['texto'] = df['texto_es']
# Solo nos quedamos con las columnas del esquema solicitado
df = df[['titulo', 'texto', 'categoria', 'autor', 'tipo']]

print("✅ Columnas consolidadas")

  0%|          | 0/343 [00:00<?, ?it/s]

  0%|          | 0/343 [00:00<?, ?it/s]


Traducciones fallidas eliminadas: 0 (Filas restantes: 343)
✅ Traducción completada y respaldo guardado.
✅ Columnas consolidadas


##  1.6.Exportación final y auditoría de calidad

In [8]:
final_df = df.copy()

# Estructura final: titulo, texto, categoria, autor, tipo
final_df = final_df[['titulo', 'texto', 'categoria', 'autor', 'tipo']]

print("=== DISTRIBUCIÓN FINAL POR CATEGORÍA ===")
print("   === DATASET - COURSERA 2021 ===")
category_counts = final_df['categoria'].value_counts()
print(category_counts)

# Alerta de umbral mínimo
print("\n--- Validación de Umbral Mínimo ---")
for cat, count in category_counts.items():
    if count < 30:
        print(f"⚠️ ADVERTENCIA: '{cat}' tiene {count} registros (Mínimo requerido: 30).")

final_df.to_csv(f'{CARPETA_PROCESADOS}/dataset_FINAL_coursera.csv', index=False)
print("\n✅ Dataset Coursera exportado con el nuevo esquema.")



display(final_df.head())

=== DISTRIBUCIÓN FINAL POR CATEGORÍA ===
   === DATASET - COURSERA 2021 ===
categoria
Backend           50
Bases de Datos    50
Cloud             50
Data Science      50
Frontend          50
Mobile            50
DevOps            43
Name: count, dtype: int64

--- Validación de Umbral Mínimo ---

✅ Dataset Coursera exportado con el nuevo esquema.


,titulo,texto,categoria,autor,tipo
0,Programación Java: construir un sistema de rec...,¿Alguna vez te has preguntado cómo decide Netf...,Backend,Duke University,articulo
1,"Comercio, inmigración y tipos de cambio en un ...",Este es el segundo de los tres cursos que form...,Backend,IE Business School,articulo
2,Comenzando con el desarrollo de aplicaciones,"En este curso, los desarrolladores de aplicaci...",Backend,Google Cloud,articulo
3,Física 101 - Fuerzas y Cinemática,Este curso sirve como una introducción a la fí...,Backend,Rice University,articulo
4,Aplicaciones Descentralizadas (Dapps),Este tercer curso de especialización en Blockc...,Backend,The State University of New York,articulo
